# 🦟 Mosquito Species Classification
### Comparative Analysis: ANN vs SVM vs Random Forest
**Classes:** Aedes · Anopheles · Culex  
**Features:** HOG + LBP (26,270-dim vector per image)  
**Dataset:** 3,000 images (1,000 per class), 224×224 px RGB

---

## ⚙️ Step 1 — Install & Import Libraries

In [ ]:
# Install required libraries (run once)
!pip install scikit-image scikit-learn opencv-python-headless matplotlib seaborn --quiet

In [ ]:
import os
import zipfile
import numpy as np
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from skimage.feature import hog, local_binary_pattern
from skimage.color import rgb2gray

from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

import time

print('✅ All libraries imported successfully!')
print(f'   OpenCV  : {cv2.__version__}')
print(f'   sklearn : {__import__("sklearn").__version__}')

## 📦 Step 2 — Upload & Extract Dataset

In [ ]:
from google.colab import files

print('📂 Please upload your archive.zip file...')
uploaded = files.upload()  # A file picker will appear — select archive.zip

zip_filename = list(uploaded.keys())[0]
print(f'\n✅ Uploaded: {zip_filename}')

In [ ]:
# Extract the zip file
EXTRACT_PATH = '/content/mosquito_dataset'

with zipfile.ZipFile(zip_filename, 'r') as z:
    z.extractall(EXTRACT_PATH)

print(f'✅ Extracted to: {EXTRACT_PATH}')

# Auto-detect the dataset root (handles nested folders inside zip)
for root, dirs, files_ in os.walk(EXTRACT_PATH):
    dirs_upper = [d.upper() for d in dirs]
    if 'AEDES' in dirs_upper and 'ANOPHELES' in dirs_upper and 'CULEX' in dirs_upper:
        DATASET_PATH = root
        break

print(f'📁 Dataset root found: {DATASET_PATH}')

# Confirm class folders and image counts
CLASSES = ['AEDES', 'ANOPHELES', 'CULEX']
for cls in CLASSES:
    folder = os.path.join(DATASET_PATH, cls)
    count = len([f for f in os.listdir(folder) if f.lower().endswith(('.jpg','.jpeg','.png'))])
    print(f'   {cls}: {count} images')

## 🔍 Step 3 — Preview Sample Images

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
fig.suptitle('Sample Images per Class', fontsize=14, fontweight='bold')

for i, cls in enumerate(CLASSES):
    folder = os.path.join(DATASET_PATH, cls)
    img_file = sorted(os.listdir(folder))[0]
    img = cv2.imread(os.path.join(folder, img_file))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    axes[i].imshow(img_rgb)
    axes[i].set_title(cls, fontsize=12, fontweight='bold')
    axes[i].axis('off')
    axes[i].set_xlabel(f'{img_rgb.shape[1]}×{img_rgb.shape[0]} px')

plt.tight_layout()
plt.savefig('sample_images.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure saved as sample_images.png')

## 🧹 Step 4 — Preprocessing & Feature Extraction

In [ ]:
# ── Feature extraction config ──────────────────────────────────────────────
IMG_SIZE     = (224, 224)   # All images are already this size, but we verify
HOG_ORIENT   = 9            # HOG: number of gradient orientation bins
HOG_PPC      = (8, 8)       # HOG: pixels per cell
HOG_CPB      = (2, 2)       # HOG: cells per block
LBP_RADIUS   = 3            # LBP: radius of circular neighborhood
LBP_POINTS   = 24           # LBP: number of sampling points (8 * radius)
LBP_METHOD   = 'uniform'    # LBP: method (uniform = rotation invariant)
LBP_BINS     = LBP_POINTS + 2  # uniform LBP → 26 histogram bins

def extract_features(img_path):
    """
    Load one image and return its combined HOG+LBP feature vector.
    Steps:
      1. Read image in BGR, convert to RGB
      2. Resize to 224×224 (verification step)
      3. Normalize pixel values to [0, 1]
      4. Extract HOG features from RGB image  → 26,244-dim
      5. Convert to grayscale for LBP
      6. Extract LBP histogram               →     26-dim
      7. Concatenate HOG + LBP              → 26,270-dim
    """
    # 1. Read & convert color space
    img_bgr = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # 2. Resize (ensures consistency even if a file is slightly different)
    img_rgb = cv2.resize(img_rgb, IMG_SIZE, interpolation=cv2.INTER_LINEAR)

    # 3. Normalize pixel values: [0,255] → [0.0, 1.0]
    img_norm = img_rgb.astype(np.float32) / 255.0

    # 4. HOG features (on normalized RGB image)
    hog_features = hog(
        img_norm,
        orientations=HOG_ORIENT,
        pixels_per_cell=HOG_PPC,
        cells_per_block=HOG_CPB,
        channel_axis=-1   # tells skimage the last axis is RGB channels
    )

    # 5. Convert to grayscale for LBP
    img_gray = rgb2gray(img_rgb)

    # 6. LBP features
    lbp_map = local_binary_pattern(
        img_gray,
        P=LBP_POINTS,
        R=LBP_RADIUS,
        method=LBP_METHOD
    )
    # Build normalized histogram of LBP codes
    lbp_hist, _ = np.histogram(
        lbp_map.ravel(),
        bins=LBP_BINS,
        range=(0, LBP_BINS),
        density=True   # normalize so sum = 1
    )

    # 7. Concatenate HOG + LBP
    return np.concatenate([hog_features, lbp_hist])

print('✅ Feature extraction function defined.')
print(f'   Expected feature vector length: HOG + LBP = 26,244 + {LBP_BINS} = {26244 + LBP_BINS}')

In [ ]:
# ── Load all images and extract features ───────────────────────────────────
print('⏳ Extracting features from all 3,000 images...')
print('   (This may take 3–7 minutes on Colab)\n')

X_raw = []   # feature vectors
y_raw = []   # string labels

start_total = time.time()

for cls in CLASSES:
    folder = os.path.join(DATASET_PATH, cls)
    img_files = sorted([
        f for f in os.listdir(folder)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ])

    start_cls = time.time()
    for fname in img_files:
        feat = extract_features(os.path.join(folder, fname))
        X_raw.append(feat)
        y_raw.append(cls)

    elapsed = time.time() - start_cls
    print(f'   ✅ {cls}: {len(img_files)} images done ({elapsed:.1f}s)')

X_raw = np.array(X_raw)
y_raw = np.array(y_raw)

total_time = time.time() - start_total
print(f'\n✅ Feature extraction complete in {total_time:.1f}s')
print(f'   Feature matrix shape: {X_raw.shape}  → (samples × features)')
print(f'   Labels shape        : {y_raw.shape}')

In [ ]:
# ── Label encoding & Min-Max scaling ───────────────────────────────────────

# Encode string labels to integers: AEDES=0, ANOPHELES=1, CULEX=2
le = LabelEncoder()
y = le.fit_transform(y_raw)
print('Label encoding:')
for i, cls in enumerate(le.classes_):
    print(f'   {cls} → {i}')

# Stratified train/test split: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y,
    test_size=0.20,
    random_state=42,
    stratify=y   # ensures equal class proportions in both sets
)

# Min-Max normalization — fit on train only, apply to both
# (fitting on test would be data leakage!)
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'\n✅ Split & scaling done')
print(f'   Training set : {X_train_scaled.shape[0]} samples')
print(f'   Test set     : {X_test_scaled.shape[0]} samples')
print(f'   Feature dims : {X_train_scaled.shape[1]}')
print(f'\nClass distribution in test set:')
for i, cls in enumerate(le.classes_):
    print(f'   {cls}: {np.sum(y_test == i)} samples')

## 🤖 Step 5 — Model Training & Hyperparameter Tuning

> ⚠️ Grid Search CV is thorough but slow. Each model may take **10–30 minutes** on Colab. Go grab a coffee ☕

### 5a — ANN (MLPClassifier)

In [ ]:
print('==='*18)
print('  TRAINING: Artificial Neural Network (ANN)')
print('==='*18)

# Baseline ANN
print('\n[1/2] Baseline ANN...')
ann_baseline = MLPClassifier(
    hidden_layer_sizes=(256, 128),
    activation='relu',
    solver='adam',
    max_iter=300,
    random_state=42
)
t0 = time.time()
ann_baseline.fit(X_train_scaled, y_train)
print(f'   Done in {time.time()-t0:.1f}s')

# Grid Search ANN (8 combos x 3-fold = 24 fits)
print('\n[2/2] Grid Search CV for ANN...')
ann_param_grid = {
    'hidden_layer_sizes': [(256, 128), (128, 64)],
    'activation'        : ['relu', 'tanh'],
    'max_iter'          : [300, 500]
}

ann_gs = GridSearchCV(
    MLPClassifier(solver='adam', learning_rate='adaptive', random_state=42),
    ann_param_grid,
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42),
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)
t0 = time.time()
ann_gs.fit(X_train_scaled, y_train)
print(f'\n   Done in {time.time()-t0:.1f}s')
print(f'   Best params : {ann_gs.best_params_}')
print(f'   Best CV F1  : {ann_gs.best_score_:.4f}')
ann_best = ann_gs.best_estimator_


### 5b — SVM (Support Vector Machine)

In [ ]:
print('==='*18)
print('  TRAINING: Support Vector Machine (SVM)')
print('==='*18)

# Baseline SVM
print('\n[1/2] Baseline SVM...')
svm_baseline = SVC(
    kernel='rbf',
    C=1.0,
    gamma='scale',
    decision_function_shape='ovo',
    random_state=42
)
t0 = time.time()
svm_baseline.fit(X_train_scaled, y_train)
print(f'   Done in {time.time()-t0:.1f}s')
print(f'   Support vectors per class: {svm_baseline.n_support_}')

# Grid Search SVM (6 combos x 3-fold = 18 fits)
print('\n[2/2] Grid Search CV for SVM...')
svm_param_grid = {
    'C'     : [1, 10, 100],
    'gamma' : ['scale', 0.01],
    'kernel': ['rbf']
}

svm_gs = GridSearchCV(
    SVC(decision_function_shape='ovo', random_state=42),
    svm_param_grid,
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42),
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)
t0 = time.time()
svm_gs.fit(X_train_scaled, y_train)
print(f'\n   Done in {time.time()-t0:.1f}s')
print(f'   Best params : {svm_gs.best_params_}')
print(f'   Best CV F1  : {svm_gs.best_score_:.4f}')
svm_best = svm_gs.best_estimator_


### 5c — Random Forest

In [ ]:
print('==='*18)
print('  TRAINING: Random Forest (RF)')
print('==='*18)

# Baseline RF
print('\n[1/2] Baseline Random Forest...')
rf_baseline = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
t0 = time.time()
rf_baseline.fit(X_train_scaled, y_train)
print(f'   Done in {time.time()-t0:.1f}s')

# Grid Search RF (12 combos x 3-fold = 36 fits)
print('\n[2/2] Grid Search CV for Random Forest...')
rf_param_grid = {
    'n_estimators' : [100, 200],
    'max_depth'    : [10, 20, None],
    'criterion'    : ['gini', 'entropy']
}

rf_gs = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    rf_param_grid,
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42),
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)
t0 = time.time()
rf_gs.fit(X_train_scaled, y_train)
print(f'\n   Done in {time.time()-t0:.1f}s')
print(f'   Best params : {rf_gs.best_params_}')
print(f'   Best CV F1  : {rf_gs.best_score_:.4f}')
rf_best = rf_gs.best_estimator_


## 📊 Step 6 — Evaluation & Results

In [ ]:
# ── Helper: evaluate one model and return metrics dict ─────────────────────
def evaluate_model(model, X_test, y_test, label):
    y_pred = model.predict(X_test)
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='macro', zero_division=0)
    rec  = recall_score(y_test, y_pred, average='macro', zero_division=0)
    f1   = f1_score(y_test, y_pred, average='macro', zero_division=0)
    return {'Model': label, 'Accuracy': acc, 'Precision': prec,
            'Recall': rec, 'F1-Score': f1, 'y_pred': y_pred}

# ── Evaluate all 6 models (3 baseline + 3 tuned) ───────────────────────────
results_baseline = [
    evaluate_model(ann_baseline, X_test_scaled, y_test, 'ANN (Baseline)'),
    evaluate_model(svm_baseline, X_test_scaled, y_test, 'SVM (Baseline)'),
    evaluate_model(rf_baseline,  X_test_scaled, y_test, 'RF  (Baseline)'),
]
results_tuned = [
    evaluate_model(ann_best, X_test_scaled, y_test, 'ANN (Tuned)'),
    evaluate_model(svm_best, X_test_scaled, y_test, 'SVM (Tuned)'),
    evaluate_model(rf_best,  X_test_scaled, y_test, 'RF  (Tuned)'),
]

all_results = results_baseline + results_tuned

# ── Print summary table ────────────────────────────────────────────────────
print('\n' + '═'*65)
print(f"  {'Model':<22} {'Accuracy':>9} {'Precision':>10} {'Recall':>8} {'F1-Score':>9}")
print('─'*65)
for r in all_results:
    print(f"  {r['Model']:<22} {r['Accuracy']:>9.4f} {r['Precision']:>10.4f} "
          f"{r['Recall']:>8.4f} {r['F1-Score']:>9.4f}")
    if r['Model'] == 'RF  (Baseline)':
        print('─'*65)
print('═'*65)

### 📈 Scenario 1 — Method Comparison (Baseline)

In [ ]:
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
models_baseline = [r['Model'] for r in results_baseline]
colors = ['#7F77DD', '#D85A30', '#3B8C2A']

x = np.arange(len(metrics))
width = 0.25

fig, ax = plt.subplots(figsize=(11, 5))
for i, (res, color) in enumerate(zip(results_baseline, colors)):
    vals = [res[m] for m in metrics]
    bars = ax.bar(x + i*width, vals, width, label=res['Model'],
                  color=color, alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xlabel('Metric', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Scenario 1 — Method Comparison (Default Hyperparameters)', fontsize=13, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.1)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('scenario1_baseline_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: scenario1_baseline_comparison.png')

### 📈 Scenario 2 — Feature Extraction Strategy Comparison

In [ ]:
# Use the best baseline model for this scenario
best_baseline_idx = np.argmax([r['F1-Score'] for r in results_baseline])
best_baseline_name = results_baseline[best_baseline_idx]['Model'].split('(')[0].strip()
print(f'Best baseline model: {best_baseline_name} — using for Scenario 2')

# Re-extract HOG-only and LBP-only features
print('\nExtracting HOG-only and LBP-only features...')
X_hog_only = []
X_lbp_only = []

for cls in CLASSES:
    folder = os.path.join(DATASET_PATH, cls)
    img_files = sorted([f for f in os.listdir(folder) if f.lower().endswith(('.jpg','.jpeg','.png'))])
    for fname in img_files:
        img_bgr = cv2.imread(os.path.join(folder, fname))
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        img_rgb = cv2.resize(img_rgb, IMG_SIZE)
        img_norm = img_rgb.astype(np.float32) / 255.0

        # HOG only
        hog_feat = hog(img_norm, orientations=HOG_ORIENT, pixels_per_cell=HOG_PPC,
                       cells_per_block=HOG_CPB, channel_axis=-1)
        X_hog_only.append(hog_feat)

        # LBP only
        img_gray = rgb2gray(img_rgb)
        lbp_map  = local_binary_pattern(img_gray, P=LBP_POINTS, R=LBP_RADIUS, method=LBP_METHOD)
        lbp_hist, _ = np.histogram(lbp_map.ravel(), bins=LBP_BINS, range=(0, LBP_BINS), density=True)
        X_lbp_only.append(lbp_hist)

X_hog_only = np.array(X_hog_only)
X_lbp_only = np.array(X_lbp_only)
print(f'HOG-only shape : {X_hog_only.shape}')
print(f'LBP-only shape : {X_lbp_only.shape}')

# Train & evaluate the best baseline model on all 3 feature sets
def train_eval_feature_set(X, y, label, ModelClass, model_params):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    scl = MinMaxScaler()
    X_tr = scl.fit_transform(X_tr)
    X_te = scl.transform(X_te)
    m = ModelClass(**model_params)
    m.fit(X_tr, y_tr)
    return evaluate_model(m, X_te, y_te, label)

model_map = {
    'ANN': (MLPClassifier, {'hidden_layer_sizes':(256,128),'activation':'relu','solver':'adam','max_iter':300,'random_state':42}),
    'SVM': (SVC,           {'kernel':'rbf','C':1.0,'gamma':'scale','decision_function_shape':'ovo','random_state':42}),
    'RF' : (RandomForestClassifier, {'n_estimators':100,'random_state':42,'n_jobs':-1}),
}
ModelClass, model_params = model_map[best_baseline_name]

s2_hog      = train_eval_feature_set(X_hog_only, y, f'HOG only ({X_hog_only.shape[1]} feats)', ModelClass, model_params)
s2_lbp      = train_eval_feature_set(X_lbp_only, y, f'LBP only ({X_lbp_only.shape[1]} feats)', ModelClass, model_params)

# FIX: directly pick the correct baseline model object — no 'and' on arrays
best_baseline_model = [ann_baseline, svm_baseline, rf_baseline][best_baseline_idx]
s2_combined = evaluate_model(best_baseline_model, X_test_scaled, y_test, f'HOG+LBP ({X_raw.shape[1]} feats)')

scenario2_results = [s2_hog, s2_lbp, s2_combined]

# Plot
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
x = np.arange(len(metrics))
width = 0.25
fig, ax = plt.subplots(figsize=(11, 5))
feat_colors = ['#5B8DB8', '#B85B5B', '#3B8C2A']
for i, (res, color) in enumerate(zip(scenario2_results, feat_colors)):
    vals = [res[m] for m in metrics]
    bars = ax.bar(x + i*width, vals, width, label=res['Model'], color=color, alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xlabel('Metric', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title(f'Scenario 2 — Feature Extraction Strategy ({best_baseline_name})', fontsize=13, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.1)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('scenario2_feature_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: scenario2_feature_comparison.png')


### 📈 Scenario 3 — Method Comparison After Hyperparameter Tuning

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
for i, (res, color) in enumerate(zip(results_tuned, colors)):
    vals = [res[m] for m in metrics]
    bars = ax.bar(x + i*width, vals, width, label=res['Model'],
                  color=color, alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xlabel('Metric', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Scenario 3 — Method Comparison (Tuned Hyperparameters)', fontsize=13, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.1)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('scenario3_tuned_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: scenario3_tuned_comparison.png')

### 📊 Confusion Matrices — All 6 Models

In [ ]:
CLASS_NAMES = le.classes_

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Confusion Matrices — Baseline (top) vs Tuned (bottom)', fontsize=14, fontweight='bold')

for idx, res in enumerate(all_results):
    row = idx // 3
    col = idx % 3
    ax  = axes[row][col]
    cm  = confusion_matrix(y_test, res['y_pred'])
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

    sns.heatmap(
        cm_pct, annot=True, fmt='.1f', ax=ax,
        cmap='Blues', cbar=False,
        xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES
    )
    ax.set_title(res['Model'], fontsize=11, fontweight='bold')
    ax.set_xlabel('Predicted', fontsize=9)
    ax.set_ylabel('Actual', fontsize=9)

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: confusion_matrices.png')

### 📋 Per-Class Classification Report (Best Tuned Model)

In [ ]:
# Find best tuned model
best_tuned_idx = np.argmax([r['F1-Score'] for r in results_tuned])
best_tuned = results_tuned[best_tuned_idx]
best_model_obj = [ann_best, svm_best, rf_best][best_tuned_idx]

print(f'\n🏆 Best Overall Model: {best_tuned["Model"]}')
print(f'   Accuracy  : {best_tuned["Accuracy"]:.4f}')
print(f'   Precision : {best_tuned["Precision"]:.4f}')
print(f'   Recall    : {best_tuned["Recall"]:.4f}')
print(f'   F1-Score  : {best_tuned["F1-Score"]:.4f}')

print('\n── Per-Class Report ──')
y_pred_best = best_model_obj.predict(X_test_scaled)
print(classification_report(y_test, y_pred_best, target_names=CLASS_NAMES))

### 📈 Baseline vs Tuned Improvement Chart

In [ ]:
model_names  = ['ANN', 'SVM', 'Random Forest']
f1_baseline  = [r['F1-Score'] for r in results_baseline]
f1_tuned     = [r['F1-Score'] for r in results_tuned]

x2 = np.arange(len(model_names))
w2 = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
b1 = ax.bar(x2 - w2/2, f1_baseline, w2, label='Baseline', color='#AAAACC', edgecolor='white')
b2 = ax.bar(x2 + w2/2, f1_tuned,    w2, label='Tuned',    color='#534AB7', edgecolor='white')

for bars in [b1, b2]:
    for bar in bars:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

# Improvement arrows
for i in range(3):
    diff = f1_tuned[i] - f1_baseline[i]
    if diff > 0:
        ax.annotate(f'+{diff:.3f}',
                    xy=(x2[i] + w2/2, f1_tuned[i] + 0.02),
                    ha='center', fontsize=8, color='green', fontweight='bold')

ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('F1-Score (Macro)', fontsize=12)
ax.set_title('F1-Score: Baseline vs Tuned — All Models', fontsize=13, fontweight='bold')
ax.set_xticks(x2)
ax.set_xticklabels(model_names)
ax.set_ylim(0, 1.1)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('baseline_vs_tuned_f1.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: baseline_vs_tuned_f1.png')

## 💾 Step 7 — Save Results & Download All Figures

In [ ]:
import csv

# Save metrics to CSV
with open('results_summary.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['Model','Accuracy','Precision','Recall','F1-Score'])
    writer.writeheader()
    for r in all_results:
        writer.writerow({k: round(v, 4) if isinstance(v, float) else v
                         for k, v in r.items() if k != 'y_pred'})

print('✅ results_summary.csv saved')

# Zip all output files for easy download
output_files = [
    'results_summary.csv',
    'sample_images.png',
    'scenario1_baseline_comparison.png',
    'scenario2_feature_comparison.png',
    'scenario3_tuned_comparison.png',
    'confusion_matrices.png',
    'baseline_vs_tuned_f1.png',
]

with zipfile.ZipFile('mosquito_results.zip', 'w') as zf:
    for fname in output_files:
        if os.path.exists(fname):
            zf.write(fname)
            print(f'   Added: {fname}')

print('\n✅ All results zipped into mosquito_results.zip')

# Download the zip
from google.colab import files
files.download('mosquito_results.zip')
print('⬇️  Download started!')